# 07 — Robot Tool & Agent Integration

The `Robot` class is a Strands `AgentTool` for controlling physical robots.
It wraps LeRobot's hardware abstraction with async task management.

## AgentTool Actions

| Action | Behavior |
|--------|----------|
| `execute` | Blocking — run until duration expires or task ends |
| `start` | Non-blocking — launch task in background thread, return immediately |
| `status` | Query current state (IDLE/CONNECTING/RUNNING/COMPLETED/STOPPED/ERROR) |
| `stop` | Interrupt a running task via shutdown event |

## Constructor

```python
from strands_robots import Robot

robot = Robot(
    tool_name="orange_arm",          # unique tool name
    robot="so101_follower",          # LeRobot config type string
    cameras={                        # camera configs → OpenCVCameraConfig
        "front": {"type": "opencv", "index_or_path": "/dev/video0", "fps": 30},
        "wrist": {"type": "opencv", "index_or_path": "/dev/video2", "fps": 30},
    },
    port="/dev/ttyACM0",             # serial port for servos
    data_config="so100_dualcam",     # GR00T data config name
    control_frequency=50.0,          # Hz — action_sleep_time = 1/freq
    action_horizon=8,                # actions consumed per inference step
)
```

`robot=` accepts: `LeRobot Robot` instance, `RobotConfig`, or type string.
String → `_create_minimal_config()` → `make_robot_from_config()`.

## With a Strands Agent

```python
from strands import Agent
from strands_robots import Robot, gr00t_inference

robot = Robot(tool_name="arm", robot="so101_follower", ...)
agent = Agent(tools=[robot, gr00t_inference])

# The agent translates natural language → tool calls:
agent("Start the arm picking up the red cube using GR00T on port 8000")
# → robot(action="start", instruction="pick up the red cube", policy_port=8000)

agent("What's the status of the arm?")
# → robot(action="status")
# → "RUNNING: 45 steps, 2.3s elapsed"

agent("Stop the arm")
# → robot(action="stop")
```

## Task State Machine

```
IDLE → CONNECTING → RUNNING → COMPLETED
                       ↓
                    STOPPED / ERROR
```

`RobotTaskState` tracks: status, instruction, start_time, duration,
step_count, error_message, and the background `Future`.

## Control Loop Internals

```python
# Inside _execute_task_async():
async def _execute_task_async(self, instruction, policy, ...):
    self._connect_robot()                          # LeRobot connect
    policy = create_policy(provider, port=...)
    policy.set_robot_state_keys(robot.joint_names)

    while not shutdown_event.is_set() and elapsed < duration:
        obs = self.robot.get_observation()          # cameras + joints
        actions = await policy.get_actions(obs, instruction)
        for action in actions[:action_horizon]:
            self.robot.send_action(action)
            await asyncio.sleep(action_sleep_time)  # 1/50 = 20ms
            step_count += 1
```

The `action_sleep_time = 1 / control_frequency` controls how fast
actions are dispatched. At 50Hz that's 20ms per action step.